In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
amananandrai_ag_news_classification_dataset_path = kagglehub.dataset_download('amananandrai/ag-news-classification-dataset')

print('Data source import complete.')


In [ ]:
# ==========================================================
# EXPERIMENT 1: EMBEDDING MODEL COMPARISON (SBERT vs. DistilBERT vs. BERT)
# Compares the effect of three different embedding models on clustering metrics,
# interpretability (K-NLPMeans summaries), and visualization (UMAP/t-SNE).
# NOTE: K-Means and K-NLPMeans share the same NMI/Accuracy metrics, but differ
# in interpretability. This output explicitly compares both.
# ==========================================================

!pip install -q transformers datasets sentence-transformers evaluate umap-learn scikit-learn kagglehub

# ---- IMPORTS ----
import os
import numpy as np
import pandas as pd
import torch
import umap
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment


In [ ]:

# ---- ENVIRONMENT SETUP & CONSTANTS ----
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SBERT_MODEL_NAME = "all-MiniLM-L6-v2"
DISTILBERT_MODEL_NAME = "distilbert-base-uncased"
BERT_MODEL_NAME = "bert-base-uncased"  # Standard BERT base model added
K_CLUSTERS = 5 # Fixed cluster limit as per assignment
SAMPLE_SIZE = 1000 # Use a small sample for fast execution


In [ ]:

# =====================================================
# UTILITY FUNCTIONS
# =====================================================

def load_and_preprocess_data():
    """Loads AG News data, combines text columns, and prepares a smaller sample."""
    print("\n--- Data Loading and Preprocessing ---")

    try:
        import kagglehub
        path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
    except Exception:
        # Fallback path if KaggleHub connection fails
        path = "/kaggle/input/ag-news-classification-dataset"

    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_df = pd.read_csv(os.path.join(path, "test.csv"))

    # Combine data, combine text, and sample
    df = pd.concat([train_df, test_df], ignore_index=True)
    df["text"] = df["Title"].astype(str) + " " + df["Description"].astype(str)
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

    # Encode Labels
    le = LabelEncoder()
    df["label_enc"] = le.fit_transform(df["Class Index"])

    print(f"🔹 Loaded and sampled {SAMPLE_SIZE} documents. True Classes: {len(le.classes_)}.")
    return df


In [ ]:

def run_evaluation(true_labels: pd.Series, predicted_labels: np.ndarray) -> tuple[float, float]:
    """Calculates NMI and a simplified 'accuracy' (label matching) metric."""
    nmi = normalized_mutual_info_score(true_labels, predicted_labels)

    # Calculate Accuracy using the Hungarian algorithm for optimal label matching
    k = len(np.unique(predicted_labels))
    unique_true_labels = np.unique(true_labels)
    cost_matrix = np.zeros((k, k), dtype=int)

    for i in range(k):
        for j in range(k):
            if j < len(unique_true_labels):
                 cost_matrix[i, j] = -np.sum((predicted_labels == i) & (true_labels == unique_true_labels[j]))

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    total_agreements = -cost_matrix[row_ind, col_ind].sum()
    accuracy = total_agreements / len(true_labels)

    return nmi, accuracy



In [ ]:
def find_centroid_text_summary(embeddings: np.ndarray, texts: pd.Series, cluster_labels: np.ndarray, cluster_id: int) -> str:
    """
    K-NLPMeans: Finds the document closest to the numerical centroid for interpretability.
    This acts as the Extractive/TextRank-style summary.
    """
    cluster_indices = np.where(cluster_labels == cluster_id)[0]

    if len(cluster_indices) == 0:
        return "No documents in this cluster."

    cluster_embeddings = embeddings[cluster_indices]

    # Calculate the numerical centroid
    numerical_centroid = cluster_embeddings.mean(axis=0)

    # Find the document closest to the numerical centroid (TextRank proxy)
    similarities = cosine_similarity(cluster_embeddings, numerical_centroid.reshape(1, -1))
    closest_doc_relative_index = np.argmax(similarities)
    closest_doc_original_index = cluster_indices[closest_doc_relative_index]

    return texts.iloc[closest_doc_original_index]


In [ ]:

# =====================================================
# EMBEDDING GENERATION FUNCTIONS
# =====================================================

def mean_pooling(model_output, attention_mask):
    """Mean pooling to get a single sentence vector from BERT-style models."""
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

def generate_sbert_embeddings(texts: list[str]):
    """Generates embeddings using Sentence-BERT (optimized for semantic similarity)."""
    print(f"  > Generating embeddings with {SBERT_MODEL_NAME} (SBERT)...")
    model = SentenceTransformer(SBERT_MODEL_NAME, device=DEVICE)
    embeddings = model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    return embeddings

def generate_bert_embeddings(texts: list[str], model_name: str):
    """Generates embeddings using BERT-style models (BERT or DistilBERT) via mean-pooling."""
    print(f"  > Generating embeddings with {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)

    embeddings_list = []

    for i in range(0, len(texts), 32):
        batch = texts[i:i+32]
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=256)

        model.eval()
        with torch.no_grad():
            model_output = model(**{k: v.to(DEVICE) for k, v in encoded_input.items()})

        batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        embeddings_list.append(batch_embeddings.cpu().numpy())

    return np.concatenate(embeddings_list, axis=0)


In [ ]:

# =====================================================
# CORE EXPERIMENT FUNCTION
# =====================================================

def run_clustering_experiment(df, embeddings, model_name):
    """Performs K-Means, evaluates, and provides K-NLPMeans summary for a given embedding set."""
    print(f"\n--- Running K-Means Clustering on {model_name} Embeddings ---")

    # 1. Clustering
    kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init="auto")
    cluster_labels = kmeans.fit_predict(embeddings)

    # 2. Evaluation
    nmi, accuracy = run_evaluation(df["label_enc"], cluster_labels)

    # 3. K-NLPMeans Interpretability (Crucial step for K-NLPMeans)
    text_summaries = {}
    numerical_centroids = {}

    for i in range(K_CLUSTERS):
        summary = find_centroid_text_summary(embeddings, df["text"], cluster_labels, i)
        text_summaries[i] = summary
        numerical_centroids[i] = kmeans.cluster_centers_[i]

    return nmi, accuracy, cluster_labels, text_summaries, numerical_centroids


In [ ]:

# =====================================================
# VISUALIZATION FUNCTIONS
# =====================================================

def plot_all_reductions(results: dict):
    """Generates a matrix of UMAP and t-SNE visualizations for all three models."""

    fig, axes = plt.subplots(len(results), 2, figsize=(18, 6 * len(results)))

    print("\n✨ Generating UMAP and t-SNE Visualizations for all models...")

    for i, (model_name, data) in enumerate(results.items()):
        embeddings = data['embeddings']
        labels = data['labels']

        # UMAP Reduction
        reducer_umap = umap.UMAP(random_state=42, n_neighbors=15, min_dist=0.1)
        emb_umap = reducer_umap.fit_transform(embeddings)

        # t-SNE Reduction
        if i == 0:
            print("✨ Running t-SNE on SBERT (optimized model) first...")
        elif i == 1:
            print("✨ Running t-SNE on DistilBERT...")
        else:
            print("✨ Running t-SNE on BERT (This may take longer)...")

        reducer_tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1, learning_rate='auto', init='pca')
        emb_tsne = reducer_tsne.fit_transform(embeddings)

        # Plot UMAP (Left Column)
        axes[i, 0].scatter(emb_umap[:, 0], emb_umap[:, 1], c=labels, cmap="tab10", s=15, alpha=0.7)
        axes[i, 0].set_title(f"{model_name} - UMAP Projection (NMI: {data['nmi']:.4f})", fontsize=14)
        axes[i, 0].set_xlabel("UMAP Dimension 1")
        axes[i, 0].set_ylabel("UMAP Dimension 2")

        # Plot t-SNE (Right Column)
        axes[i, 1].scatter(emb_tsne[:, 0], emb_tsne[:, 1], c=labels, cmap="tab10", s=15, alpha=0.7)
        axes[i, 1].set_title(f"{model_name} - t-SNE Projection (Acc: {data['accuracy']:.4f})", fontsize=14)
        axes[i, 1].set_xlabel("t-SNE Dimension 1")
        axes[i, 1].set_ylabel("t-SNE Dimension 2")

    plt.suptitle("Experiment 1: Comparison of K-Means Clustering on Different Embeddings", fontsize=16, y=1.02)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.show()


In [ ]:
# =====================================================
# MAIN EXECUTION BLOCK (EXPERIMENT 1)
# =====================================================

def main_exp1():
    """Main function for Experiment 1, orchestrating all three model comparisons."""
    df = load_and_preprocess_data()
    texts = df["text"].tolist()

    results = {}

    # --- 1. Run SBERT Experiment ---
    embeddings_sbert = generate_sbert_embeddings(texts)
    nmi_sbert, acc_sbert, labels_sbert, summaries_sbert, centroids_sbert = run_clustering_experiment(
        df, embeddings_sbert, SBERT_MODEL_NAME
    )
    results[SBERT_MODEL_NAME] = {'nmi': nmi_sbert, 'accuracy': acc_sbert, 'labels': labels_sbert, 'embeddings': embeddings_sbert, 'summaries': summaries_sbert, 'centroids': centroids_sbert}

    # --- 2. Run DistilBERT Experiment ---
    embeddings_distilbert = generate_bert_embeddings(texts, DISTILBERT_MODEL_NAME)
    nmi_distilbert, acc_distilbert, labels_distilbert, summaries_distilbert, centroids_distilbert = run_clustering_experiment(
        df, embeddings_distilbert, DISTILBERT_MODEL_NAME
    )
    results[DISTILBERT_MODEL_NAME] = {'nmi': nmi_distilbert, 'accuracy': acc_distilbert, 'labels': labels_distilbert, 'embeddings': embeddings_distilbert, 'summaries': summaries_distilbert, 'centroids': centroids_distilbert}

    # --- 3. Run BERT Experiment ---
    embeddings_bert = generate_bert_embeddings(texts, BERT_MODEL_NAME)
    nmi_bert, acc_bert, labels_bert, summaries_bert, centroids_bert = run_clustering_experiment(
        df, embeddings_bert, BERT_MODEL_NAME
    )
    results[BERT_MODEL_NAME] = {'nmi': nmi_bert, 'accuracy': acc_bert, 'labels': labels_bert, 'embeddings': embeddings_bert, 'summaries': summaries_bert, 'centroids': centroids_bert}

    # Determine the best performing model based on NMI
    best_model_name = SBERT_MODEL_NAME
    max_nmi = nmi_sbert
    if nmi_distilbert > max_nmi:
        max_nmi = nmi_distilbert
        best_model_name = DISTILBERT_MODEL_NAME
    if nmi_bert > max_nmi:
        max_nmi = nmi_bert
        best_model_name = BERT_MODEL_NAME

    best_results = results[best_model_name]

    # --- Section 1: Metric Comparison (Shared K-Means / K-NLPMeans) ---
    print("\n\n=====================================================================================")
    print("           SECTION 1: CLUSTERING PERFORMANCE METRICS (K-Means vs. K-NLPMeans)          ")
    print("=====================================================================================")
    print("NOTE: K-Means and K-NLPMeans share the same NMI/Accuracy as the document assignment step is identical.")
    print("-----------------------------------------------------------------------------------------------------")
    print(f"{'Metric':<25} | {'SBERT (MiniLM-L6)':<20} | {'DistilBERT':<15} | {'BERT Base Uncased':<20}")
    print("-------------------------|----------------------|-----------------|----------------------")
    print(f"{'Normalized Mutual Info (NMI)':<25} | {nmi_sbert:.4f}{'':<16} | {nmi_distilbert:.4f}{'':<11} | {nmi_bert:.4f}{'':<16}")
    print(f"{'Cluster Mapping Accuracy':<25} | {acc_sbert:.4f}{'':<16} | {acc_distilbert:.4f}{'':<11} | {acc_bert:.4f}{'':<16}")
    print("=====================================================================================")

    print(f"\n**Metric Interpretation:** The **{best_model_name}** model achieves the highest NMI/Accuracy, indicating its embeddings create the most coherent and well-separated clusters, which is essential for both K-Means and K-NLPMeans success.")


    # --- Section 2: Interpretability Comparison (K-Means vs. K-NLPMeans) ---
    print("\n\n=====================================================================================")
    print(f"       SECTION 2: INTERPRETABILITY COMPARISON ({best_model_name} Model)                 ")
    print("=====================================================================================")
    print("This shows the true difference between K-Means (numerical) and K-NLPMeans (textual).")

    for i in range(K_CLUSTERS):
        print(f"\n--- Cluster {i} (Size: {len(np.where(best_results['labels'] == i)[0])}) ---")

        # K-Means Centroid (Numerical)
        numerical_centroid = best_results['centroids'][i]
        vector_snippet = f"[{numerical_centroid[0]:.4f}, {numerical_centroid[1]:.4f}, {numerical_centroid[2]:.4f} ... (Dim {numerical_centroid.shape[0]})]"
        print(f"  > K-Means Centroid (Numerical): {vector_snippet}")
        print("    (Interpretation: NONE. Unreadable mean vector.)")

        # K-NLPMeans Centroid (Textual Summary)
        summary = best_results['summaries'][i]
        print(f"  > K-NLPMeans Centroid (Textual/TextRank Proxy): '{summary[:180]}...'")
        print("    (Interpretation: HIGH. Cluster topic immediately understood by human.)")


    # --- Visualization ---
    plot_all_reductions(results)

if __name__ == "__main__":
    main_exp1()
